In [1]:
text = "low low lower lowest"

print(text)

low low lower lowest


In [2]:
chars = sorted((set(text)))

stoi = {
    ch:i
    for i,ch in enumerate(chars)
}

itos = {
    i:ch
    for i,ch in enumerate(chars)
}

tokens = [stoi[ch] for ch in text]

print("vocab: ",chars)
print("token ids: ",tokens)

vocab:  [' ', 'e', 'l', 'o', 'r', 's', 't', 'w']
token ids:  [2, 3, 7, 0, 2, 3, 7, 0, 2, 3, 7, 1, 4, 0, 2, 3, 7, 1, 5, 6]


In [3]:
from collections import Counter

pairs = Counter(zip(tokens,tokens[1:]))
print("most common pairs:")

for pair,count in pairs.most_common():
    print(pair,count)

most common pairs:
(2, 3) 4
(3, 7) 4
(0, 2) 3
(7, 0) 2
(7, 1) 2
(1, 4) 1
(4, 0) 1
(1, 5) 1
(5, 6) 1


In [4]:
for (a,b),count in pairs.most_common():
    print(f"('{itos[a]}','{itos[b]})-> {count}")
    

('l','o)-> 4
('o','w)-> 4
(' ','l)-> 3
('w',' )-> 2
('w','e)-> 2
('e','r)-> 1
('r',' )-> 1
('e','s)-> 1
('s','t)-> 1


In [5]:
def merge_pair(tokens,pair,new_token_id):
    new_tokens = []
    i = 0

    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i],tokens[i+1]) == pair:
            new_tokens.append(new_token_id)
            i+=2
        else:
            new_tokens.append(tokens[i])
            i+=1

    return new_tokens

In [6]:
lo_id = len(stoi)
stoi["lo"] = lo_id
itos[lo_id] = 'lo'
print(lo_id)

8


In [7]:
tokens = merge_pair(tokens,(stoi["l"],stoi["o"]),lo_id)

print(tokens)

[8, 7, 0, 8, 7, 0, 8, 7, 1, 4, 0, 8, 7, 1, 5, 6]


In [8]:
print([itos[token] for token in tokens])

['lo', 'w', ' ', 'lo', 'w', ' ', 'lo', 'w', 'e', 'r', ' ', 'lo', 'w', 'e', 's', 't']


In [9]:
low_id = len(stoi)
stoi["low"] = low_id
itos[low_id] = "low"

print(low_id)

9


In [10]:
tokens = merge_pair(
    tokens,
    (stoi["lo"], stoi["w"]),
    low_id
)
print(tokens)
print((stoi["lo"], stoi["w"]))
print([itos[token] for token in tokens])

[9, 0, 9, 0, 9, 1, 4, 0, 9, 1, 5, 6]
(8, 7)
['low', ' ', 'low', ' ', 'low', 'e', 'r', ' ', 'low', 'e', 's', 't']


In [11]:
print("lo ID:", stoi["lo"])
print("w ID:", stoi["w"])
print("low ID:", low_id)

lo ID: 8
w ID: 7
low ID: 9


In [12]:
print([(itos[tokens[i]], itos[tokens[i+1]]) 
       for i in range(len(tokens)-1)])

[('low', ' '), (' ', 'low'), ('low', ' '), (' ', 'low'), ('low', 'e'), ('e', 'r'), ('r', ' '), (' ', 'low'), ('low', 'e'), ('e', 's'), ('s', 't')]


In [51]:
from collections import Counter


def train_bpe(text, num_merges):

    # character vocabulary
    vocab = sorted(set(text))

    stoi = {ch: i for i, ch in enumerate(vocab)}
    itos = {i: ch for i, ch in enumerate(vocab)}

    # turn text into IDs
    tokens = [stoi[ch] for ch in text]

    merges = {}

    for merge_num in range(num_merges):

        # count pairs
        pairs = Counter(zip(tokens, tokens[1:]))

        if not pairs:
            break

        # most common pair
        pair, count = pairs.most_common(1)[0]

        # new token ID
        new_token_id = len(itos)

        # create the new token
        new_token = itos[pair[0]] + itos[pair[1]]

        # add it to both vocabularies
        stoi[new_token] = new_token_id
        itos[new_token_id] = new_token

        # merge the pair
        new_tokens = []
        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_token_id)
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

        merges[pair] = new_token_id

        print(
            f"Merge {merge_num + 1}: "
            f"'{itos[pair[0]]}' + '{itos[pair[1]]}' "
            f"-> '{new_token}' ({count} times)"
        )

    return tokens, stoi, itos, merges

In [18]:
tokens, stoi, itos, merges = train_bpe(
    "low low lower lowest",
    num_merges=5
)
print([itos[token] for token in tokens])

Merge 1: 'l' + 'o' -> 'lo' (4 times)
Merge 2: 'lo' + 'w' -> 'low' (4 times)
Merge 3: ' ' + 'low' -> ' low' (3 times)
Merge 4: ' low' + 'e' -> ' lowe' (2 times)
Merge 5: 'low' + ' low' -> 'low low' (1 times)
['low low', ' lowe', 'r', ' lowe', 's', 't']


In [19]:
def encode(text, stoi, merges):
    tokens = [stoi[ch] for ch in text]

    for pair, new_token_id in merges.items():

        new_tokens = []
        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_token_id)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens

In [20]:
encoded = encode("lowest", stoi, merges)

print(encoded)
print([itos[token] for token in encoded])

[9, 1, 5, 6]
['low', 'e', 's', 't']


In [21]:
def decode(tokens, itos):
    return "".join(itos[token] for token in tokens)

In [22]:
decoded = decode(encoded, itos)

print(decoded)

lowest


In [64]:
from collections import Counter


def train_byte_bpe(text, num_merges):

    # start with all 256 possible bytes
    stoi = {bytes([i]): i for i in range(256)}
    itos = {i: bytes([i]) for i in range(256)}

    # convert text to bytes
    tokens = list(text.encode("utf-8"))

    merges = {}

    for merge_num in range(num_merges):

        # count adjacent pairs
        pairs = Counter(zip(tokens, tokens[1:]))

        if not pairs:
            break

        # find the most common pair
        pair, count = pairs.most_common(1)[0]

        # create a new token
        new_token_id = len(itos)
        new_token = itos[pair[0]] + itos[pair[1]]

        # add the new token
        stoi[new_token] = new_token_id
        itos[new_token_id] = new_token

        # merge the pair
        new_tokens = []
        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_token_id)
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

        # remember the merge
        merges[pair] = {
            "id": new_token_id,
            "rank": merge_num
        }

        print(
            f"Merge {merge_num + 1}: "
            f"{itos[pair[0]]} + {itos[pair[1]]} "
            f"-> {new_token} ({count} times)"
        )

    return tokens, stoi, itos, merges

In [71]:
def encode(text, stoi, merges):

    # start with raw bytes
    tokens = list(text.encode("utf-8"))

    while True:

        # find all pairs currently present
        pairs = list(zip(tokens, tokens[1:]))

        # keep only pairs that the tokenizer knows how to merge
        available = [
            pair for pair in pairs
            if pair in merges
        ]

        # nothing left to merge
        if not available:
            break

        # choose the highest-priority merge
        pair = min(
            available,
            key=lambda p: merges[p]["rank"]
        )

        new_token_id = merges[pair]["id"]

        # apply that one merge
        new_tokens = []
        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_token_id)
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens

In [54]:
def decode(tokens, itos):

    # turn token IDs back into bytes
    byte_data = b"".join(itos[token] for token in tokens)

    # turn bytes back into normal text
    return byte_data.decode("utf-8")

In [72]:
text = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune.
"""

In [73]:
tokens, stoi, itos, merges = train_byte_bpe(
    text,
    num_merges=20
)

Merge 1: b' ' + b't' -> b' t' (5 times)
Merge 2: b'h' + b'e' -> b'he' (5 times)
Merge 3: b's' + b' ' -> b's ' (4 times)
Merge 4: b'o' + b' ' -> b'o ' (3 times)
Merge 5: b'r' + b' ' -> b'r ' (3 times)
Merge 6: b'he' + b' ' -> b'he ' (3 times)
Merge 7: b'i' + b'n' -> b'in' (3 times)
Merge 8: b'\n' + b'T' -> b'\nT' (2 times)
Merge 9: b'o ' + b'b' -> b'o b' (2 times)
Merge 10: b'o b' + b'e' -> b'o be' (2 times)
Merge 11: b'o be' + b',' -> b'o be,' (2 times)
Merge 12: b' ' + b'o' -> b' o' (2 times)
Merge 13: b'n' + b'o' -> b'no' (2 times)
Merge 14: b' t' + b'he ' -> b' the ' (2 times)
Merge 15: b't' + b'i' -> b'ti' (2 times)
Merge 16: b'\nT' + b'o be,' -> b'\nTo be,' (1 times)
Merge 17: b'\nTo be,' + b' o' -> b'\nTo be, o' (1 times)
Merge 18: b'\nTo be, o' + b'r ' -> b'\nTo be, or ' (1 times)
Merge 19: b'\nTo be, or ' + b'no' -> b'\nTo be, or no' (1 times)
Merge 20: b'\nTo be, or no' + b't' -> b'\nTo be, or not' (1 times)


In [77]:
text = "To be, or not to be"

encoded = encode(text, stoi, merges)

print(encoded)
print([itos[token] for token in encoded])

[84, 266, 267, 260, 268, 116, 256, 265]
[b'T', b'o be,', b' o', b'r ', b'no', b't', b' t', b'o be']


In [78]:
decoded = decode(encoded, itos)

print(decoded)
print(text == decoded)

To be, or not to be
True


In [76]:
text = "to be or not to be"

encoded = encode(text, stoi, merges)

print("Encoded:")
print(encoded)

decoded = decode(encoded, itos)

print("\nDecoded:")
print(decoded)

Encoded:
[116, 265, 267, 260, 268, 116, 256, 265]

Decoded:
to be or not to be


In [57]:
text = "Hello नमस्ते 🚀"

encoded = encode(text, stoi, merges)
decoded = decode(encoded, itos)

print("Encoded:", encoded)
print("Decoded:", decoded)
print("Same:", text == decoded)

Encoded: [72, 257, 256, 32, 224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135, 32, 240, 159, 154, 128]
Decoded: Hello नमस्ते 🚀
Same: True


In [39]:
from collections import Counter

class BPETokenizer:

    def __init__(self):
        self.stoi = {}
        self.itos = {}
        self.merges = {}

    def train(self,text,num_merges):
        # start with characters
        vocab = sorted(set(text))

        self.stoi = {ch: i for i, ch in enumerate(vocab)}
        self.itos = {i: ch for i, ch in enumerate(vocab)}

        # token for characters we don't know
        unk_id = len(self.itos)

        self.stoi["<UNK>"] = unk_id
        self.itos[unk_id] = "<UNK>"

        tokens = [self.stoi[ch] for ch in text]

        for merge_num in range(num_merges):

            # count adjacent pairs
            pairs = Counter(zip(tokens, tokens[1:]))

            if not pairs:
                break

            # most common pair
            pair, count = pairs.most_common(1)[0]

            # new token
            new_token_id = len(self.itos)
            new_token = self.itos[pair[0]] + self.itos[pair[1]]

            # add it to the vocabulary
            self.stoi[new_token] = new_token_id
            self.itos[new_token_id] = new_token

            # merge the pair
            new_tokens = []
            i = 0

            while i < len(tokens):

                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                    new_tokens.append(new_token_id)
                    i += 2

                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens

            # remember the merge
            self.merges[pair] = new_token_id

            print(
                f"Merge {merge_num + 1}: "
                f"'{self.itos[pair[0]]}' + '{self.itos[pair[1]]}' "
                f"-> '{new_token}' ({count} times)"
            )


    def encode(self, text):

        tokens = [self.stoi.get(ch, self.stoi["<UNK>"]) for ch in text]

        for pair, new_token_id in self.merges.items():

            new_tokens = []
            i = 0

            while i < len(tokens):

                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                    new_tokens.append(new_token_id)
                    i += 2

                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens

        return tokens

    def decode(self, tokens):

        return "".join(self.itos[token] for token in tokens)

In [41]:
tokenizer = BPETokenizer()

In [30]:
tokenizer.train(
    "low low lower lowest",
    num_merges=5
)

Merge 1: 'l' + 'o' -> 'lo' (4 times)
Merge 2: 'lo' + 'w' -> 'low' (4 times)
Merge 3: ' ' + 'low' -> ' low' (3 times)
Merge 4: ' low' + 'e' -> ' lowe' (2 times)
Merge 5: 'low' + ' low' -> 'low low' (1 times)


In [31]:
encoded = tokenizer.encode("lowest")

print(encoded)
print(tokenizer.decode(encoded))

[10, 1, 5, 6]
lowest


In [34]:
hello = "Hello"
enc_hello = tokenizer.encode(hello)
print(tokenizer.decode(enc_hello))

<UNK>ello


In [35]:
text = "Hello"

byte_values = list(text.encode("utf-8"))

print(byte_values)

[72, 101, 108, 108, 111]


In [36]:
text = "नमस्ते"

byte_values = list(text.encode("utf-8"))

print(byte_values)

[224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135]


# connecting it to gpt

In [80]:
from google.colab import drive

drive.mount('/content/drive')


with open(
    "/content/drive/MyDrive/tiny_llm/input.txt",
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [81]:
print(len(text))
print(text[:200])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [82]:
bpe_text = text[:100_000]

print(len(bpe_text))

100000


In [83]:
tokens, stoi, itos, merges = train_byte_bpe(
    bpe_text,
    num_merges=500
)

Merge 1: b'e' + b' ' -> b'e ' (2734 times)
Merge 2: b't' + b'h' -> b'th' (2054 times)
Merge 3: b't' + b' ' -> b't ' (1441 times)
Merge 4: b's' + b' ' -> b's ' (1383 times)
Merge 5: b'o' + b'u' -> b'ou' (1327 times)
Merge 6: b',' + b' ' -> b', ' (1160 times)
Merge 7: b'd' + b' ' -> b'd ' (1133 times)
Merge 8: b'e' + b'r' -> b'er' (997 times)
Merge 9: b'a' + b'n' -> b'an' (858 times)
Merge 10: b'i' + b'n' -> b'in' (834 times)
Merge 11: b':' + b'\n' -> b':\n' (820 times)
Merge 12: b' ' + b'th' -> b' th' (811 times)
Merge 13: b'e' + b'n' -> b'en' (774 times)
Merge 14: b'\n' + b'\n' -> b'\n\n' (735 times)
Merge 15: b'o' + b'r' -> b'or' (703 times)
Merge 16: b'o' + b'n' -> b'on' (674 times)
Merge 17: b'y' + b' ' -> b'y ' (659 times)
Merge 18: b'a' + b'r' -> b'ar' (637 times)
Merge 19: b'o' + b' ' -> b'o ' (603 times)
Merge 20: b'h' + b'a' -> b'ha' (575 times)
Merge 21: b'l' + b'l' -> b'll' (569 times)
Merge 22: b'y' + b'ou' -> b'you' (548 times)
Merge 23: b'.' + b'\n\n' -> b'.\n\n' (519 time

In [84]:
print("Vocabulary size:", len(itos))
print("Number of merges:", len(merges))

Vocabulary size: 756
Number of merges: 500


In [85]:
for token_id in range(256, min(276, len(itos))):
    print(token_id, itos[token_id])

256 b'e '
257 b'th'
258 b't '
259 b's '
260 b'ou'
261 b', '
262 b'd '
263 b'er'
264 b'an'
265 b'in'
266 b':\n'
267 b' th'
268 b'en'
269 b'\n\n'
270 b'or'
271 b'on'
272 b'y '
273 b'ar'
274 b'o '
275 b'ha'


In [86]:
sample = text[:200]

encoded = encode(sample, stoi, merges)
decoded = decode(encoded, itos)

print("Original:")
print(sample)

print("\nDecoded:")
print(decoded)

print("\nSame:", sample == decoded)

Original:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you

Decoded:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you

Same: True


In [87]:
print("Characters:", len(sample))
print("BPE tokens:", len(encoded))

Characters: 200
BPE tokens: 71
